In [ ]:
!which python
!python --version

/home/commander-data/Storage/users/mave/.venv/bin/python
Python 3.12.9


In [ ]:
import pandas as pd
import gffutils
import os

<module 'functions' from '/home/commander-data/Storage/users/mave/search_sequences/library_design/functions.py'>

In [ ]:
gene = {
    "symbol": "NF1",
    "ensembl_gene_id": "ENSG00000196712",
    "chrom": "chr17",
    "start": 31094977,
    "end": 31377675,
    "mane_select": "ENST00000358273", # version 9
    "refseq_select": "NM_001042492", # version 3
    "strand": "+"
}

In [ ]:
## set paths
basepath = "/home/commander-data/Storage/users/mave"
genome_data_path = basepath + "/genome_data"
transcripts_path = genome_data_path + "/ensembl_transcripts/115"
ensembl_gff_db = transcripts_path + '/Homo_sapiens.GRCh38.115.db' # create the database using create_gff_db.py script
genome_path = genome_data_path + "/genomes/grch38.fa"

# gene specific gff paths
transcripts_oi_path = genome_data_path + "/genes"
os.makedirs(transcripts_oi_path, exist_ok=True)
transcripts_oi_file = transcripts_oi_path + "/ensembl_" + gene["symbol"] + ".gff3"

# splice prediction paths
work_path = basepath + "/search_sequences"
datapath = work_path + "/data/library_design"
pangolin_scores_path = datapath + "/splicing_prediction_exons_consequence.vcf"
spliceai_scores_path = datapath + "/splicing_prediction_exons_spliceai.vcf"

In [ ]:
# get exons
ensembl_gff = gffutils.FeatureDB(ensembl_gff_db, keep_order=True)

exons_oi = None
for exon_id, cds in enumerate(ensembl_gff.children('transcript:' + gene["mane_select"], featuretype="CDS", order_by="start")):
    start = cds.start
    end = cds.end
    chrom = cds.chrom
    new_data = pd.DataFrame([chrom, start, end, exon_id + 1]).T

    if exons_oi is None:
        exons_oi = new_data
    else:
        exons_oi = pd.concat([exons_oi, new_data], axis = 0)
exons_oi.columns = ["chrom", "start", "end", "exon_id"]

In [ ]:
# computed in library_design.ipynb
library = pd.read_csv(basepath + "/search_sequences/library_design/design_sequences/data/variants.tsv", sep = "\t")

In [ ]:
prio_exon_lib = library.copy()
prio_exon_lib = prio_exon_lib.groupby("start").apply(lambda x: x if x["pangolin_gain"].max() > 0.2 else None).reset_index(drop = True)
tmp = prio_exon_lib[((prio_exon_lib["pangolin_gain"] > 0.2) & (abs(prio_exon_lib["pangolin_loss"]) < abs(prio_exon_lib["pangolin_gain"])) & (prio_exon_lib["splice_effect_to_canon_ss_dist"] > 10))].copy() #  
positions_oi = tmp["start"].unique()
prio_exon_lib = prio_exon_lib[prio_exon_lib["start"].isin(positions_oi)]

prio_exon_lib["is_pangolin"] = prio_exon_lib["pangolin_gain"] > 0.2
prio_exon_lib["is_clinvar"] = prio_exon_lib["clinvar"] != "unknown"

prio_exon_lib = prio_exon_lib[prio_exon_lib["is_pangolin"] | prio_exon_lib["is_clinvar"]]

prio_exon_lib

/tmp/ipykernel_3225182/4227121636.py:2: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  prio_exon_lib = prio_exon_lib.groupby("start").apply(lambda x: x if x["pangolin_gain"].max() > 0.2 else None).reset_index(drop = True)


,var_id,pangolin_gain,pangolin_gain_pos,pangolin_loss,pangolin_loss_pos,consequence,spliceai_gain,spliceai_gain_pos,spliceai_loss,spliceai_loss_pos,...,chrom,start,ref,len_ref,alt,len_alt,end,splice_effect_to_canon_ss_dist,is_pangolin,is_clinvar
0,chr17-31156021-A-G,0.70,0,-0.16,105,synonymous_variant,0.00,91,-0.07,105,...,chr17,31156021,A,1,G,1,31156021,38,True,True
3,chr17-31156024-C-T,0.03,-3,-0.00,-41,synonymous_variant,0.00,88,-0.00,-384,...,chr17,31156024,C,1,T,1,31156024,38,False,True
4,chr17-31156024-C-A,0.82,-3,-0.49,102,synonymous_variant,0.00,88,-0.33,102,...,chr17,31156024,C,1,A,1,31156024,38,True,True
5,chr17-31156024-C-G,0.68,-3,-0.12,102,synonymous_variant,0.00,88,-0.06,102,...,chr17,31156024,C,1,G,1,31156024,38,True,False
7,chr17-31156044-A-T,0.88,-2,-0.79,82,missense_variant,0.00,333,-0.86,82,...,chr17,31156044,A,1,T,1,31156044,59,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1142,chr17-31360539-T-A,0.22,2,-0.06,-52,stop_gained,0.21,2,-0.00,63,...,chr17,31360539,T,1,A,1,31360539,54,True,False
1143,chr17-31360548-C-T,0.00,80,-0.00,-61,missense_variant,0.00,-280,0.00,155,...,chr17,31360548,C,1,T,1,31360548,75,False,True
1144,chr17-31360548-C-G,0.80,1,-0.35,-61,missense_variant,0.96,1,-0.00,54,...,chr17,31360548,C,1,G,1,31360548,62,True,False
1145,chr17-31360548-C-A,0.00,-61,-0.00,6,missense_variant,0.00,2,0.00,155,...,chr17,31360548,C,1,A,1,31360548,0,False,True


In [ ]:
exon_scores = prio_exon_lib.groupby("exon_number").apply(lambda x: len(x["start"].unique())).copy()
exon_scores = exon_scores.reset_index()
exon_scores.columns = ["exon_id", "num_vars"]
exon_scores = exon_scores.merge(exons_oi, on = "exon_id")
exon_scores["exon_length"] = abs(exon_scores["start"] - exon_scores["end"])
exon_scores["score"] = (exon_scores["num_vars"]) / (exon_scores["exon_length"])
exon_scores = exon_scores.sort_values("score", ascending = False)
exon_scores

#17,25,34,39,42

/tmp/ipykernel_3225182/2612187692.py:28: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  exon_scores = data.groupby("exon_number").apply(lambda x: len(x["start"].unique()))


,exon_id,num_vars,chrom,start,end,exon_length,score
10,17.0,23,17,31225095,31225250,155,0.148387
17,25.0,13,17,31232073,31232189,116,0.112069
26,34.0,12,17,31260369,31260515,146,0.082192
29,37.0,31,17,31325820,31326252,432,0.071759
31,39.0,10,17,31330296,31330498,202,0.049505
34,42.0,12,17,31336635,31336914,279,0.043011
9,16.0,5,17,31223444,31223567,123,0.04065
6,11.0,3,17,31201411,31201485,74,0.040541
4,9.0,7,17,31200422,31200595,173,0.040462
13,21.0,17,17,31229025,31229465,440,0.038636
